# Módulo 5b — Implementación LoRa en SDR

Este notebook implementa el sistema LoRa completo sobre hardware real usando
el SDR ADALM-PLUTO.



## Contenido

1. Verificación de conexión al PlutoSDR
2. Configuración del hardware
3. Loopback digital (sin radio)
4. Loopback por antena (con cable TX→RX)
5. Transmisión entre dos SDRs (TX y RX separados)
6. Curva BER vs potencia de transmisión
7. Transmisión del mensaje "¡Hola Com Dig!"


## 1. Librerías e imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import adi                    # librería del PlutoSDR (pyadi-iio)
import time

# ============================================================
# Importamos todas las funciones del Módulo 5a
# (copiadas acá para que el notebook sea autónomo)
# ============================================================

# ---------- Módulo 1 ----------
def codificador(bits, SF):
    bits = np.array(bits)
    if len(bits) % SF != 0:
        raise ValueError(f"Cantidad de bits ({len(bits)}) debe ser múltiplo de SF ({SF})")
    num_simbolos = len(bits) // SF
    simbolos = np.zeros(num_simbolos, dtype=int)
    for i in range(num_simbolos):
        bloque = bits[i * SF : (i + 1) * SF]
        valor = 0
        for posicion, bit in enumerate(bloque):
            valor += bit * 2 ** (SF - 1 - posicion)
        simbolos[i] = valor
    return simbolos

def decodificador(simbolos, SF):
    simbolos = np.array(simbolos, dtype=int)
    bits = np.zeros(len(simbolos) * SF, dtype=int)
    for i, simbolo in enumerate(simbolos):
        valor_restante = int(simbolo)
        for posicion in range(SF):
            peso = 2 ** (SF - 1 - posicion)
            if valor_restante >= peso:
                bits[i * SF + posicion] = 1
                valor_restante -= peso
    return bits

def calcular_ber(bits_tx, bits_rx):
    bits_tx = np.array(bits_tx); bits_rx = np.array(bits_rx)
    if len(bits_tx) != len(bits_rx):
        raise ValueError("Los vectores deben tener la misma longitud")
    return np.sum(bits_tx != bits_rx) / len(bits_tx)

# ---------- Módulo 2 ----------
def up_chirp_base(SF, BW, Fs):
    N = 2**SF
    chirp = np.zeros(N, dtype=complex)
    for k in range(N):
        chirp[k] = np.exp(1j * 2 * np.pi * (k**2 / (2*N) - k/2))
    return chirp

def down_chirp(SF, BW, Fs):
    return np.conj(up_chirp_base(SF, BW, Fs))

def waveform_former(simbolos, SF, BW, Fs):
    N = 2**SF
    simbolos = np.array(simbolos)
    cb = up_chirp_base(SF, BW, Fs)
    wf = np.zeros((len(simbolos), N), dtype=complex)
    for i, s in enumerate(simbolos):
        wf[i] = np.roll(cb, -s)
    return wf

# ---------- Módulo 5a ----------
NUM_PREAMBLE = 8

def peak_merging(espectro):
    N = len(espectro); max_suma = -1; bin_detectado = 0
    for k in range(N):
        kn = (k + 1) % N
        suma = espectro[k] + espectro[kn]
        if suma > max_suma:
            max_suma = suma
            bin_detectado = k if espectro[k] >= espectro[kn] else kn
    return bin_detectado

def generar_trama(simbolos_datos, SF, BW, Fs, silencio_simbolos=5):
    N = 2**SF
    uc = up_chirp_base(SF, BW, Fs)
    dc = down_chirp(SF, BW, Fs)
    silencio = np.zeros(silencio_simbolos * N, dtype=complex)
    preamble = np.tile(uc, NUM_PREAMBLE)
    sfd      = np.concatenate([dc, dc, dc[:N//4]])
    payload  = waveform_former(simbolos_datos, SF, BW, Fs).flatten()
    return np.concatenate([silencio, preamble, sfd, payload])

def detectar_preamble(señal, SF, BW, Fs, tolerancia=2):
    N = 2**SF; dc = down_chirp(SF, BW, Fs)
    umbral = 0.1; cons = 0; bin_ref = -1
    for i in range(len(señal) // N):
        v = señal[i*N:(i+1)*N]
        if np.mean(np.abs(v)**2) < umbral:
            cons = 0; bin_ref = -1; continue
        esp = np.abs(np.fft.fft(v * dc))
        ba  = peak_merging(esp)
        if bin_ref == -1:
            bin_ref = ba; cons = 1
        elif min(abs(ba-bin_ref), N-abs(ba-bin_ref)) <= tolerancia:
            cons += 1
        else:
            bin_ref = ba; cons = 1
        if cons >= NUM_PREAMBLE:
            return (i + 1) * N, bin_ref
    return -1, -1

def alinear_ventana(señal, idx_fin_preamble, bin_preamble, SF, BW, Fs):
    N = 2**SF
    idx_payload = idx_fin_preamble + int(2.25 * N)
    cfo_bins = bin_preamble if bin_preamble <= N//2 else bin_preamble - N
    return idx_payload, cfo_bins

def demodular_trama(señal, num_simbolos, SF, BW, Fs):
    N = 2**SF; dc = down_chirp(SF, BW, Fs)
    idx_fin, bin_preamble = detectar_preamble(señal, SF, BW, Fs)
    if idx_fin == -1:
        return None, {'error': 'Preámbulo no detectado'}
    idx_payload, cfo_bins = alinear_ventana(
        señal, idx_fin, bin_preamble, SF, BW, Fs)
    simbolos_rx = np.zeros(num_simbolos, dtype=int)
    for i in range(num_simbolos):
        inicio = idx_payload + i * N
        fin    = inicio + N
        if fin > len(señal):
            simbolos_rx[i] = 0; continue
        esp = np.abs(np.fft.fft(señal[inicio:fin] * dc))
        simbolos_rx[i] = (peak_merging(esp) - cfo_bins) % N
    info = {'idx_fin_preamble': idx_fin, 'idx_payload': idx_payload,
            'bin_preamble': bin_preamble, 'cfo_bins': cfo_bins,
            'cfo_hz': cfo_bins * BW / N}
    return simbolos_rx, info

def texto_a_bits(texto):
    bits = []
    for c in texto:
        v = ord(c)
        for i in range(7, -1, -1):
            bits.append((v >> i) & 1)
    return np.array(bits)

def bits_a_texto(bits):
    texto = ''
    for i in range(len(bits) // 8):
        byte = bits[i*8:(i+1)*8]
        v = sum(b * 2**(7-j) for j, b in enumerate(byte))
        try:    texto += chr(v)
        except: texto += '?'
    return texto

print("Todas las funciones cargadas correctamente.")

## 2. Parámetros del sistema y configuración del SDR

Los parámetros de RF del PlutoSDR:
- **Frecuencia de portadora (fc):** 926 MHz — dentro de la banda ISM, libre de licencia
- **Sample rate:** debe ser ≥ BW. Usamos 521 kHz (≥ 4× BW) para dar margen al filtro interno del Pluto
- **Atenuación TX:** controla la potencia de transmisión. Valores más negativos = menos potencia


In [ ]:
# ---------- Parámetros LoRa ----------
SF  = 7
BW  = 125e3       # Ancho de banda LoRa
Fs  = BW          # Frecuencia de muestreo de la señal banda base
N   = 2**SF

# ---------- Parámetros RF del PlutoSDR ----------
FC          = 926e6     # Frecuencia de portadora: 926 MHz (banda ISM)
SAMPLE_RATE = 521e3     # Sample rate del ADC/DAC del Pluto (≥ BW)
TX_ATTEN    = -30       # Atenuación TX en dB (rango: 0 a -89 dB)
                        # -30 dB es un valor moderado para loopback
RX_GAIN     = 30        # Ganancia RX en dB

# Factor de resampleo: la señal banda base está a Fs=125kHz
# pero el Pluto trabaja a SAMPLE_RATE=521kHz
# Hay que interpolar la señal al sample rate del Pluto antes de transmitir
RESAMPLE_FACTOR = int(SAMPLE_RATE / Fs)  # = 4

print(f"=== Parámetros del sistema ===")
print(f"SF              : {SF}")
print(f"BW              : {BW/1e3:.0f} kHz")
print(f"Fs (banda base) : {Fs/1e3:.0f} kHz")
print(f"Fc              : {FC/1e6:.0f} MHz")
print(f"Sample rate     : {SAMPLE_RATE/1e3:.0f} kHz")
print(f"Resample factor : {RESAMPLE_FACTOR}x")
print(f"TX atenuación   : {TX_ATTEN} dB")

## 3. Funciones de resampleo

El PlutoSDR trabaja a `SAMPLE_RATE` (521 kHz) pero nuestra señal está a `Fs` (125 kHz).
Antes de transmitir hay que **interpolar** (subir el sample rate) y después de recibir
hay que **decimar** (bajar el sample rate) para volver a banda base.

La interpolación más simple es repetir cada muestra `RESAMPLE_FACTOR` veces.


In [ ]:
def interpolar(señal, factor):
    """
    Sube el sample rate repitiendo cada muestra 'factor' veces.
    Convierte la señal de Fs a Fs*factor.
    """
    return np.repeat(señal, factor)

def decimar(señal, factor):
    """
    Baja el sample rate tomando 1 de cada 'factor' muestras.
    Convierte la señal de Fs*factor a Fs.
    """
    return señal[::factor]

def normalizar(señal):
    """
    Normaliza la señal para que su valor máximo sea 1.0
    (requerido por el PlutoSDR para evitar saturación del DAC).
    """
    max_val = np.max(np.abs(señal))
    if max_val > 0:
        return señal / max_val
    return señal

print("Funciones de resampleo definidas.")

## 4. Verificación de conexión al PlutoSDR

Antes de transmitir verificamos que el Pluto está conectado y responde.
La IP por defecto del Pluto es `192.168.2.1` cuando se conecta por USB.

> **Si la siguiente celda da error:** verificá que el Pluto esté encendido
> y conectado. En el JupyterHub del lab debería estar disponible
> en la IP configurada por el laboratorio.


In [ ]:
# Dirección IP del PlutoSDR en el laboratorio
PLUTO_IP = "ip:192.168.2.1"   # Modificar si el lab usa otra IP

try:
    sdr = adi.Pluto(PLUTO_IP)
    print(f"✅ PlutoSDR conectado correctamente")
    print(f"   URI: {PLUTO_IP}")
except Exception as e:
    print(f"❌ Error al conectar con el PlutoSDR: {e}")
    print(f"   Verificá que el Pluto esté encendido y en {PLUTO_IP}")
    print(f"   En el lab puede ser otra IP — consultá al docente")

## 5. Configuración del PlutoSDR

Configuramos los parámetros de TX y RX del hardware.


In [ ]:
def configurar_sdr(sdr, fc, sample_rate, tx_atten, rx_gain):
    """
    Configura el PlutoSDR para transmisión y recepción LoRa.

    Parámetros
    ----------
    sdr         : objeto adi.Pluto ya conectado
    fc          : frecuencia de portadora en Hz
    sample_rate : tasa de muestreo del ADC/DAC en Hz
    tx_atten    : atenuación TX en dB (negativo, ej: -30)
    rx_gain     : ganancia RX en dB
    """
    # Frecuencia de portadora (igual para TX y RX)
    sdr.tx_lo = int(fc)
    sdr.rx_lo = int(fc)

    # Sample rate del ADC y DAC
    sdr.sample_rate = int(sample_rate)

    # Ancho de banda de los filtros internos
    sdr.tx_rf_bandwidth = int(sample_rate)
    sdr.rx_rf_bandwidth = int(sample_rate)

    # Control de ganancia
    sdr.tx_hardwaregain_chan0 = tx_atten   # TX: negativo = menos potencia
    sdr.gain_control_mode_chan0 = 'manual'
    sdr.rx_hardwaregain_chan0 = rx_gain    # RX: ganancia manual

    # Cantidad de muestras por buffer de recepción
    sdr.rx_buffer_size = 4 * 2**SF * int(sample_rate / (BW))

    print(f"✅ SDR configurado:")
    print(f"   Fc          = {fc/1e6:.0f} MHz")
    print(f"   Sample rate = {sample_rate/1e3:.0f} kHz")
    print(f"   TX atenuación = {tx_atten} dB")
    print(f"   RX ganancia   = {rx_gain} dB")

configurar_sdr(sdr, FC, SAMPLE_RATE, TX_ATTEN, RX_GAIN)

## 6. Funciones de transmisión y recepción

Estas funciones encapsulan el ciclo completo TX/RX con el PlutoSDR.


In [ ]:
def transmitir(sdr, trama, resample_factor):
    """
    Transmite una trama LoRa por el PlutoSDR.

    1. Interpola la señal al sample rate del Pluto
    2. Normaliza la amplitud
    3. Envía al hardware (tx_cyclic_buffer para transmisión continua)
    """
    # Interpolamos la señal banda base al sample rate del Pluto
    señal_rf = interpolar(trama, resample_factor)

    # Normalizamos para no saturar el DAC
    señal_rf = normalizar(señal_rf)

    # El Pluto espera la señal como array de tipo complex64
    señal_rf = señal_rf.astype(np.complex64)

    # Activamos modo cíclico: repite la señal continuamente
    sdr.tx_cyclic_buffer = True
    sdr.tx(señal_rf)

def recibir(sdr, resample_factor, num_buffers=3):
    """
    Recibe muestras del PlutoSDR y las decima al sample rate de banda base.

    Descarta los primeros buffers (pueden contener transitorios de inicio).
    """
    muestras_totales = np.array([], dtype=complex)

    for i in range(num_buffers):
        buffer = sdr.rx()
        if i > 0:   # Descartamos el primer buffer (transitorio)
            muestras_totales = np.concatenate([muestras_totales, buffer])

    # Decimamos de vuelta al sample rate de banda base
    señal_bb = decimar(muestras_totales, resample_factor)

    return señal_bb.astype(complex)

def detener_tx(sdr):
    """Detiene la transmisión y limpia el buffer TX."""
    sdr.tx_destroy_buffer()

print("Funciones TX/RX definidas.")

## 7. Loopback digital

El loopback digital conecta el transmisor con el receptor **por software**:
la señal se genera, se pasa por el pipeline de recepción, y se demodula
sin pasar por el hardware de radio. Es exactamente lo que ya probamos
en el `05a_Simulacion.ipynb`, pero acá lo corremos en el entorno del lab
para confirmar que las funciones funcionan antes de encender el Pluto.


In [ ]:
np.random.seed(42)

mensaje    = "Hola Com Dig!"
bits_msg   = texto_a_bits(mensaje)
resto      = len(bits_msg) % SF
if resto:
    bits_msg = np.concatenate([bits_msg, np.zeros(SF - resto, dtype=int)])

simbolos_tx = codificador(bits_msg, SF)
trama_tx    = generar_trama(simbolos_tx, SF, BW, Fs)

# Loopback digital: la señal recibida es idéntica a la transmitida
trama_rx    = trama_tx.copy()

simbolos_rx, info = demodular_trama(trama_rx, len(simbolos_tx), SF, BW, Fs)
bits_rx     = decodificador(simbolos_rx, SF)
msg_rx      = bits_a_texto(bits_rx[:len(mensaje)*8])

print("=== Loopback digital ===")
print(f"Mensaje TX : '{mensaje}'")
print(f"Mensaje RX : '{msg_rx}'")
print(f"BER        : {calcular_ber(bits_msg[:len(mensaje)*8], bits_rx[:len(mensaje)*8]):.4f}")
print(f"CFO est.   : {info['cfo_hz']:.1f} Hz")
print(f"Estado     : {'✅ OK' if msg_rx == mensaje else '❌ Error'}")

## 8. Loopback por antena

En el loopback por antena se conecta físicamente el puerto TX con el RX
del mismo PlutoSDR usando un cable coaxial SMA-SMA.

La señal sí pasa por el hardware (DAC → cable → ADC), lo que permite
detectar problemas del hardware sin necesitar propagación por el aire.

> **Conexión física requerida:** cable SMA entre TX y RX del mismo Pluto.
> Usar un atenuador de al menos 20-30 dB entre TX y RX para no saturar el RX.


In [ ]:
print("=== Loopback por antena ===")
print("Verificá que el cable SMA esté conectando TX con RX del PlutoSDR.")
print()

try:
    # Generamos la trama
    np.random.seed(10)
    bits_lb    = np.random.randint(0, 2, 20 * SF)
    sim_lb     = codificador(bits_lb, SF)
    trama_lb   = generar_trama(sim_lb, SF, BW, Fs)

    # Transmitimos
    transmitir(sdr, trama_lb, RESAMPLE_FACTOR)
    time.sleep(0.1)   # pequeña espera para que el Pluto estabilice

    # Recibimos
    señal_rx = recibir(sdr, RESAMPLE_FACTOR, num_buffers=4)
    detener_tx(sdr)

    # Normalizamos la señal recibida
    señal_rx = señal_rx / np.max(np.abs(señal_rx))

    # Demodulamos
    sim_rx, info = demodular_trama(señal_rx, len(sim_lb), SF, BW, Fs)

    if sim_rx is None:
        print("❌ Preámbulo no detectado")
        print("   Causas posibles: señal muy débil, cable mal conectado,")
        print("   o ganancia RX muy baja. Ajustá TX_ATTEN o RX_GAIN.")
    else:
        bits_rx = decodificador(sim_rx, SF)
        ber     = calcular_ber(bits_lb, bits_rx)
        print(f"✅ Trama detectada")
        print(f"   CFO estimado : {info['cfo_hz']:.1f} Hz")
        print(f"   BER          : {ber:.4f}")
        print(f"   Símbolos TX  : {sim_lb[:5]} ...")
        print(f"   Símbolos RX  : {sim_rx[:5]} ...")

except Exception as e:
    print(f"❌ Error durante el loopback por antena: {e}")

## 9. Transmisión entre dos SDRs

Para transmisión real entre dos PlutoSDRs, uno actúa como TX y el otro como RX.
El JupyterHub del lab puede tener ambos disponibles en IPs distintas.

Modificá las IPs según la configuración del laboratorio.


In [ ]:
# IPs de los dos PlutoSDRs (modificar según el lab)
PLUTO_TX_IP = "ip:192.168.2.1"
PLUTO_RX_IP = "ip:192.168.2.2"

print("Para transmisión entre dos SDRs:")
print(f"  SDR TX: {PLUTO_TX_IP}")
print(f"  SDR RX: {PLUTO_RX_IP}")
print()
print("Verificá las IPs con el docente del laboratorio antes de correr esta celda.")
print("Una vez confirmadas las IPs, descomentá y ejecutá el bloque siguiente.")

In [ ]:
# ============================================================
# DESCOMENTÁ ESTE BLOQUE CUANDO ESTÉS EN EL LAB CON 2 SDRs
# ============================================================

# try:
#     sdr_tx = adi.Pluto(PLUTO_TX_IP)
#     sdr_rx = adi.Pluto(PLUTO_RX_IP)
#     configurar_sdr(sdr_tx, FC, SAMPLE_RATE, TX_ATTEN, RX_GAIN)
#     configurar_sdr(sdr_rx, FC, SAMPLE_RATE, TX_ATTEN, RX_GAIN)
#     print("✅ Ambos SDRs configurados")
#
#     np.random.seed(20)
#     bits_2sdr   = np.random.randint(0, 2, 30 * SF)
#     sim_2sdr    = codificador(bits_2sdr, SF)
#     trama_2sdr  = generar_trama(sim_2sdr, SF, BW, Fs)
#
#     transmitir(sdr_tx, trama_2sdr, RESAMPLE_FACTOR)
#     time.sleep(0.2)
#     señal_rx_2sdr = recibir(sdr_rx, RESAMPLE_FACTOR, num_buffers=5)
#     detener_tx(sdr_tx)
#
#     señal_rx_2sdr = señal_rx_2sdr / np.max(np.abs(señal_rx_2sdr))
#     sim_rx_2sdr, info_2sdr = demodular_trama(señal_rx_2sdr, len(sim_2sdr), SF, BW, Fs)
#
#     if sim_rx_2sdr is None:
#         print("❌ Preámbulo no detectado en RX")
#     else:
#         bits_rx_2sdr = decodificador(sim_rx_2sdr, SF)
#         ber_2sdr = calcular_ber(bits_2sdr, bits_rx_2sdr)
#         print(f"✅ Transmisión exitosa entre 2 SDRs")
#         print(f"   BER    : {ber_2sdr:.4f}")
#         print(f"   CFO    : {info_2sdr['cfo_hz']:.1f} Hz")
#
# except Exception as e:
#     print(f"❌ Error: {e}")

print("Bloque de 2 SDRs listo para descomentar en el lab.")

## 10. Curva BER vs atenuación TX

Medimos la BER variando la atenuación del transmisor.
Mayor atenuación = menos potencia = más errores.

Esta curva es el equivalente de la BER vs SNR pero medida con hardware real,
donde el "SNR" se controla indirectamente a través de la potencia de TX.


In [ ]:
# ============================================================
# CORRER EN EL LAB con el loopback por antena conectado
# ============================================================

atenuaciones = [-10, -20, -30, -40, -50, -60]
ber_hw = []

print(f"{'TX Atten (dB)':>15} | {'BER':>10} | {'Sync':>6}")
print("-" * 38)

try:
    for atten in atenuaciones:
        sdr.tx_hardwaregain_chan0 = atten

        np.random.seed(99)
        bits_test = np.random.randint(0, 2, 30 * SF)
        sim_test  = codificador(bits_test, SF)
        trama_test = generar_trama(sim_test, SF, BW, Fs)

        transmitir(sdr, trama_test, RESAMPLE_FACTOR)
        time.sleep(0.1)
        señal_rx_t = recibir(sdr, RESAMPLE_FACTOR, num_buffers=4)
        detener_tx(sdr)

        señal_rx_t = señal_rx_t / np.max(np.abs(señal_rx_t))
        sim_rx_t, info_t = demodular_trama(señal_rx_t, len(sim_test), SF, BW, Fs)

        if sim_rx_t is None:
            ber_hw.append(0.5)
            print(f"{atten:>15} | {'—':>10} | {'❌':>6}")
        else:
            bits_rx_t = decodificador(sim_rx_t, SF)
            ber_t = calcular_ber(bits_test, bits_rx_t)
            ber_hw.append(ber_t)
            print(f"{atten:>15} | {ber_t:>10.4f} | {'✅':>6}")

    ber_hw = np.array(ber_hw)

    # Graficamos
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.semilogy(atenuaciones, np.where(np.array(ber_hw) > 0, ber_hw, 1e-5),
                marker='o', markersize=6, linewidth=1.5, color='steelblue')
    ax.set_xlabel("Atenuación TX (dB)")
    ax.set_ylabel("BER")
    ax.set_title("BER vs Atenuación TX — Loopback por antena (Hardware real)")
    ax.grid(True, which='both', alpha=0.4)
    ax.invert_xaxis()   # Menos atenuación = más potencia (izquierda)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"❌ Error durante el barrido: {e}")
    print("   Verificá la conexión del loopback por antena.")

## 11. Transmisión del mensaje "¡Hola Com Dig!" con hardware real

La prueba final: transmitir el mensaje por el aire (o por cable en loopback)
y verificar que llega correctamente.


In [ ]:
mensaje_final = "Hola Com Dig!"
print(f"=== Transmitiendo: '{mensaje_final}' ===")
print()

try:
    bits_final = texto_a_bits(mensaje_final)
    resto = len(bits_final) % SF
    if resto:
        bits_final = np.concatenate([bits_final, np.zeros(SF - resto, dtype=int)])

    sim_final  = codificador(bits_final, SF)
    trama_final = generar_trama(sim_final, SF, BW, Fs)

    print(f"Caracteres : {len(mensaje_final)}")
    print(f"Bits       : {len(bits_final)}")
    print(f"Símbolos   : {len(sim_final)}")
    print(f"Muestras   : {len(trama_final)}")
    print()

    # Transmitimos
    sdr.tx_hardwaregain_chan0 = -20   # Potencia moderada
    transmitir(sdr, trama_final, RESAMPLE_FACTOR)
    time.sleep(0.15)

    # Recibimos
    señal_rx_f = recibir(sdr, RESAMPLE_FACTOR, num_buffers=5)
    detener_tx(sdr)
    señal_rx_f = señal_rx_f / np.max(np.abs(señal_rx_f))

    # Demodulamos
    sim_rx_f, info_f = demodular_trama(señal_rx_f, len(sim_final), SF, BW, Fs)

    if sim_rx_f is None:
        print("❌ Preámbulo no detectado — señal demasiado débil o ruidosa")
    else:
        bits_rx_f   = decodificador(sim_rx_f, SF)
        msg_recibido = bits_a_texto(bits_rx_f[:len(mensaje_final)*8])
        ber_f = calcular_ber(bits_final[:len(mensaje_final)*8],
                              bits_rx_f[:len(mensaje_final)*8])

        print(f"Mensaje TX     : '{mensaje_final}'")
        print(f"Mensaje RX     : '{msg_recibido}'")
        print(f"BER            : {ber_f:.4f}")
        print(f"CFO estimado   : {info_f['cfo_hz']:.1f} Hz")
        estado = '✅ Mensaje recibido correctamente' if msg_recibido == mensaje_final else '❌ Errores en la recepción'
        print(f"Estado         : {estado}")

except Exception as e:
    print(f"❌ Error: {e}")

## 12. Espectrograma de la señal recibida

Visualizamos la señal que llegó al receptor para confirmar que
la estructura de la trama (preámbulo, SFD, payload) es visible.


In [ ]:
try:
    if 'señal_rx_f' in dir() and señal_rx_f is not None:
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.specgram(señal_rx_f, NFFT=N//4, Fs=Fs, noverlap=N//8,
                    cmap='inferno', sides='twosided')
        ax.set_title("Espectrograma de la señal recibida por el PlutoSDR")
        ax.set_xlabel("Tiempo (s)")
        ax.set_ylabel("Frecuencia (Hz)")
        ax.set_ylim(-BW/2*1.1, BW/2*1.1)
        ax.axhline(-BW/2, color='cyan', linestyle='--', linewidth=0.7)
        ax.axhline( BW/2, color='cyan', linestyle='--', linewidth=0.7)
        plt.tight_layout()
        plt.show()
    else:
        print("Corré primero la celda de transmisión del mensaje (sección 11).")
except Exception as e:
    print(f"Error al graficar: {e}")

## 13. Conclusiones del módulo

Este notebook implementa el sistema LoRa completo sobre hardware real:

- **Loopback digital:** confirmación de que el pipeline de software funciona
  correctamente en el entorno del laboratorio.
- **Loopback por antena:** validación del hardware (DAC → cable → ADC)
  sin depender de la propagación por el aire.
- **Transmisión entre 2 SDRs:** prueba de comunicación real por el aire,
  incluyendo todos los efectos de canal reales (multipath, CFO, ruido).
- **Curva BER vs atenuación:** caracterización de la robustez del sistema
  en función de la potencia de transmisión.
- **Mensaje "¡Hola Com Dig!":** demostración extremo a extremo del sistema
  completo: texto → bits → chirps → radio → chirps → bits → texto.

